## Supplied-data update — 4 November 2026
This notebook retains its previously executed analysis and source-access record. The subsequently uploaded BSOS CSV and codebooks are now analysed separately in **06_BSOS_Anxiety_Measurement.ipynb** and Appendix J. Earlier statements that BSOS data or empirical GAD-7 scores were unavailable are historical, not the current project status. The new file still lacks personalisation, comparison and digital-literacy variables. It is not merged with this notebook’s respondents.

# Notebook 05 Statistical and measurement audit

The Impact of Perceived Social Media Algorithm Personalisation on Anxiety Levels Among Generation Z: The Mediating Role of Social Comparison

This secondary-only supplement adds full ordinal predictor reporting, likelihood-based fit, and the diary within-person fit increment. The original samples, prediction models and held-out evaluation remain unchanged. **It does not estimate the approved mediation pathway or empirical GAD-7 scores.** Those require an authorised dataset with the necessary jointly observed measurements. The decision record was written after the original results were known; these are exploratory checks.

## 1 Prepare a CPU runtime

Open this .ipynb in Google Colab and run the cells in order. The analysis requires the original Ahmed CSV and Taylor–Chen Study 2 workbook. Obtain them through Notebooks 02 and 04, or upload the original files below. A checksum failure means that the source needs review; do not remove the check. No new participant data are collected.

In [1]:
import sys, subprocess
from importlib.metadata import version, PackageNotFoundError
required={"numpy":"2.3.5","pandas":"2.2.3","scipy":"1.17.0","statsmodels":"0.15.0","openpyxl":"3.1.5"}
missing=[]
for package,wanted in required.items():
    try:
        if version(package)!=wanted: missing.append(f"{package}=={wanted}")
    except PackageNotFoundError: missing.append(f"{package}=={wanted}")
if missing:
    subprocess.check_call([sys.executable,"-m","pip","install","-q",*missing])
    raise RuntimeError("Dependencies installed. Restart the runtime, then run all cells again.")
print("Dependency versions verified.")

Dependency versions verified.


## 2 Locate the verified original sources

Ahmed source: https://www.kaggle.com/datasets/souvikahmed071/social-media-and-mental-health

Taylor and Chen Study 2: https://osf.io/cs2aj/ — workbook https://osf.io/download/qkr8s/ and codebook https://osf.io/download/atnk8/

In a fresh Colab runtime, upload smmh.csv and lonelyalgo_study2.xlsx when requested. The two sources remain separate and are not merged.

In [2]:
from pathlib import Path
IN_COLAB=Path("/content").exists()
ROOT=Path("/content/statistical_audit") if IN_COLAB else Path.cwd()/"final_revision"
ROOT.mkdir(parents=True,exist_ok=True)
SURVEY=Path("/content/research_project/data/smmh.csv") if IN_COLAB else Path.cwd()/"data/smmh.csv"
DIARY=Path("/content/research_extension/lonelyalgo_study2.xlsx") if IN_COLAB else Path.cwd()/"revision/lonelyalgo_study2.xlsx"
if IN_COLAB and (not SURVEY.exists() or not DIARY.exists()):
    from google.colab import files
    print("Upload the two original files named above; no rows will be printed.")
    uploaded=files.upload()
    for filename,target in [("smmh.csv",SURVEY),("lonelyalgo_study2.xlsx",DIARY)]:
        if not target.exists() and filename in uploaded:
            target.parent.mkdir(parents=True,exist_ok=True)
            target.write_bytes(uploaded[filename])
    del uploaded
if not SURVEY.exists() or not DIARY.exists():
    raise FileNotFoundError("Required original files are missing. Run Notebooks 02 and 04 or upload the exact files. No statistical analysis has run.")
OUT=ROOT/"results"
OUT.mkdir(exist_ok=True)
print("Source paths available; checksum verification follows.")

Source paths available; checksum verification follows.


## 3 Define the analyses

The ordinal model keeps all five worry categories. McFadden pseudo R-squared compares log-likelihoods and is not a percentage of anxiety variance. The focal comparison test is separated from twelve exploratory secondary coefficient tests, which receive Holm correction. Diary inference clusters repeated observations by participant. Incremental within-person R-squared describes fit on observed complete days, not a causal algorithm contribution.

In [3]:
"""Reproduce additional diagnostics without changing the original analyses.

Usage: python statistical_audit.py --survey PATH --diary PATH --out DIRECTORY
No respondent-level outputs are exported. All added tests are exploratory.
"""
from pathlib import Path
import argparse
import hashlib
import json
import platform
import numpy as np
import pandas as pd
import scipy
from scipy.stats import chi2
import statsmodels
import statsmodels.formula.api as smf
from statsmodels.miscmodels.ordinal_model import OrderedModel
from statsmodels.stats.multitest import multipletests

SURVEY_SHA = 'fd7943ecf6e9b8dbaf678556a7e9413d26509b9e9be99b59406a64a482bd3d52'
DIARY_SHA = '3370d21ad5bd6bd53e33ac21208ab2bbf1ae6479c50045cee1817edcc25de590'
NAMES = ['timestamp','age','gender','relationship','occupation','affiliation','uses_social','platforms','time_category','purposeless','distraction_social','restlessness','distractibility','worry','concentration','comparison','comparison_feeling','validation','low_mood','interest_fluctuation','sleep']

def verified(path, expected):
    assert hashlib.sha256(Path(path).read_bytes()).hexdigest() == expected, 'Source bytes changed: audit before analysing.'

def survey_audit(path):
    verified(path, SURVEY_SHA)
    a = pd.read_csv(path)
    assert a.shape == (481,21)
    a.columns = NAMES
    for p in ['TikTok','Instagram','YouTube']:
        a[p] = a.platforms.str.split(',').apply(lambda v:int(p in [x.strip() for x in v]))
    a = a[a.age.between(18,24) & a.age.eq(np.floor(a.age)) & a.uses_social.eq('Yes') & a[['TikTok','Instagram','YouTube']].any(axis=1)].copy()
    assert len(a) == 295
    a['student'] = a.occupation.isin(['University Student','School Student']).astype(int)
    a['gender_group'] = a.gender.str.strip().map({'Female':'Female','Male':'Male'}).fillna('Another or unspecified response')
    order = ['More than 5 hours','Less than an Hour','Between 1 and 2 hours','Between 2 and 3 hours','Between 3 and 4 hours','Between 4 and 5 hours']
    x = a[['age','student','TikTok','Instagram','YouTube']].astype(float)
    x = x.join(pd.get_dummies(pd.Series(pd.Categorical(a.time_category,categories=order),index=a.index),prefix='duration',drop_first=True,dtype=float))
    x = x.join(pd.get_dummies(pd.Series(pd.Categorical(a.gender_group,categories=['Female','Male','Another or unspecified response']),index=a.index),prefix='gender',drop_first=True,dtype=float))
    x['comparison'] = a.comparison.astype(float)
    y = a.worry.astype(int)
    full = OrderedModel(y,x,distr='logit').fit(method='bfgs',disp=False,maxiter=2000)
    base = OrderedModel(y,x.drop(columns='comparison'),distr='logit').fit(method='bfgs',disp=False,maxiter=2000)
    assert full.mle_retvals['converged'] and base.mle_retvals['converged']
    counts = y.value_counts().sort_index()
    null_ll = float((counts*np.log(counts/len(y))).sum())
    rows=[]
    for label,fit in [('Without comparison',base),('With comparison',full)]:
        rows.append({'model':label,'n':len(y),'parameters_including_thresholds':len(fit.params),'log_likelihood':fit.llf,'null_log_likelihood':null_ll,'mcfadden_pseudo_r2':1-fit.llf/null_ll,'aic':fit.aic,'bic':fit.bic})
    fit_table=pd.DataFrame(rows)
    ci=full.conf_int()
    coeff=pd.DataFrame([{'term':v,'log_odds':full.params[v],'odds_ratio':np.exp(full.params[v]),'ci_low':np.exp(ci.loc[v,0]),'ci_high':np.exp(ci.loc[v,1]),'p_unadjusted':full.pvalues[v]} for v in x.columns])
    other=coeff.term.ne('comparison')
    coeff['p_holm_secondary']=np.nan
    coeff.loc[other,'p_holm_secondary']=multipletests(coeff.loc[other,'p_unadjusted'],method='holm')[1]
    lr=2*(full.llf-base.llf)
    info={'n':len(y),'comparison_lr':float(lr),'comparison_lr_df':1,'comparison_lr_p':float(chi2.sf(lr,1)),
          'mcfadden_increment':float(fit_table.iloc[1].mcfadden_pseudo_r2-fit_table.iloc[0].mcfadden_pseudo_r2),
          'full_vs_null_lr':float(2*(full.llf-null_ll)),'full_vs_null_df':len(x.columns),
          'full_vs_null_p':float(chi2.sf(2*(full.llf-null_ll),len(x.columns))),
          'references':{'gender':'Female','duration':'More than 5 hours','student':'Other occupation','platforms':'Non-use of that platform'},
          'meaning':'McFadden pseudo R squared is likelihood-based fit, not variance explained or a causal effect.',
          'coefficient_test_family':'Comparison focal; other 12 slopes Holm-adjusted as an exploratory family.'}
    assert np.isclose(np.exp(full.params['comparison']),1.729296,rtol=1e-4)
    return info,fit_table,coeff

def diary_audit(path):
    verified(path, DIARY_SHA)
    d=pd.read_excel(path)
    assert d.shape==(4060,11) and not d.duplicated(['id','day']).any()
    q=d.dropna(subset=['connect','par','pai','tkbrowse','unusual']).copy()
    n=q.groupby('id').size()
    q=q[q.id.isin(n[n>=2].index)].copy()
    assert (len(q),q.id.nunique())==(2466,276)
    full=smf.ols('connect ~ par + pai + tkbrowse + unusual + C(day) + C(id)',data=q).fit()
    base=smf.ols('connect ~ tkbrowse + unusual + C(day) + C(id)',data=q).fit()
    robust=full.get_robustcov_results(cov_type='cluster',groups=q.id,use_correction=True,use_t=True)
    within_y=q.connect-q.groupby('id').connect.transform('mean')
    tss=float(within_y@within_y)
    r2_full=1-full.ssr/tss
    r2_base=1-base.ssr/tss
    restriction=np.zeros((2,len(full.params)))
    for row,v in enumerate(['par','pai']): restriction[row,list(full.params.index).index(v)]=1
    joint=robust.f_test(restriction)
    info={'n_person_days':len(q),'people':int(q.id.nunique()),'within_r2_full':float(r2_full),'within_r2_without_perceptions':float(r2_base),'within_r2_increment':float(r2_full-r2_base),'perception_partial_r2':float((base.ssr-full.ssr)/base.ssr),'joint_cluster_f':float(np.asarray(joint.fvalue).item()),'joint_df_num':float(joint.df_num),'joint_df_denom':float(joint.df_denom),'joint_p':float(joint.pvalue),'definition':'R squared uses within-person demeaned connectedness variation. Increment adds PAR and PAI jointly to browsing, event and day effects. Not anxiety variance or causal attribution.'}
    assert np.isclose(full.params['par'],.071673,rtol=1e-4)
    return info

def score_verified_gad7(frame, columns, *, instrument_verified=False, coding_verified=False, timeframe_verified=False):
    """Complete-item scoring only; flags must refer to inspected source documents.

    This utility does not establish validity or diagnose anyone. It rejects
    incomplete or non-integer/out-of-range coding rather than inventing items.
    Missing items yield a missing total, never a zero-filled clinical score.
    """
    if not all([instrument_verified,coding_verified,timeframe_verified]):
        raise ValueError('Verify the instrument, 0–3 response coding and two-week timeframe in the source first.')
    if len(columns)!=7 or len(set(columns))!=7 or not set(columns)<=set(frame.columns):
        raise ValueError('Seven distinct, documented GAD-7 item columns are required.')
    items=frame[columns].apply(pd.to_numeric,errors='raise')
    observed=items.stack()
    if not observed.isin([0,1,2,3]).all():
        raise ValueError('Unexpected GAD-7 item coding. Do not guess a recoding.')
    return items.sum(axis=1,min_count=7).rename('gad7_total')



## 4 Run and export aggregate results

In [4]:
survey,fit_table,coefficient_table=survey_audit(SURVEY)
diary=diary_audit(DIARY)
fit_table.to_csv(OUT/"ordinal_model_fit.csv",index=False)
coefficient_table.to_csv(OUT/"all_ordinal_predictors.csv",index=False)
results={"status":"EXECUTED_EXPLORATORY_DIAGNOSTICS","survey":survey,"diary":diary,"original_mediation_status":"NOT_ESTIMATED_MISSING_JOINT_MEASURES","gad7_empirical_status":"NOT_EXECUTED_NO_AUTHORISED_VERIFIED_ITEM_DATA","software":{"python":platform.python_version(),"numpy":np.__version__,"pandas":pd.__version__,"scipy":scipy.__version__,"statsmodels":statsmodels.__version__}}
(OUT/"statistical_audit.json").write_text(json.dumps(results,indent=2))
from IPython.display import display
display(fit_table.round(5))
display(coefficient_table.round(5))
print(json.dumps(results,indent=2))

{
  "status": "EXECUTED_EXPLORATORY_DIAGNOSTICS",
  "survey": {
    "n": 295,
    "comparison_lr": 42.58757373660933,
    "comparison_lr_df": 1,
    "comparison_lr_p": 6.758739630857756e-11,
    "mcfadden_increment": 0.05021035974295529,
    "full_vs_null_lr": 67.40665615124931,
    "full_vs_null_df": 13,
    "full_vs_null_p": 2.401121479476924e-09,
    "references": {
      "gender": "Female",
      "duration": "More than 5 hours",
      "student": "Other occupation",
      "platforms": "Non-use of that platform"
    },
    "meaning": "McFadden pseudo R squared is likelihood-based fit, not variance explained or a causal effect.",
    "coefficient_test_family": "Comparison focal; other 12 slopes Holm-adjusted as an exploratory family."
  },
  "diary": {
    "n_person_days": 2466,
    "people": 276,
    "within_r2_full": 0.020118705999418673,
    "within_r2_without_perceptions": 0.01575699599341973,
    "within_r2_increment": 0.004361710005998942,
    "perception_partial_r2": 0.00443153

,model,n,parameters_including_thresholds,log_likelihood,null_log_likelihood,mcfadden_pseudo_r2,aic,bic
0,Without comparison,295,16,-411.68196,-424.0915,0.02926,855.36392,914.35553
1,With comparison,295,17,-390.38817,-424.0915,0.07947,814.77635,877.45493


,term,log_odds,odds_ratio,ci_low,ci_high,p_unadjusted,p_holm_secondary
0,age,-0.09368,0.91057,0.78069,1.06207,0.23283,1.00000
1,student,-0.26621,0.76628,0.28134,2.08711,0.60256,1.00000
2,TikTok,0.25794,1.29427,0.76283,2.19593,0.33892,1.00000
3,Instagram,0.45326,1.57344,0.80839,3.06251,0.18222,1.00000
4,YouTube,-0.05252,0.94884,0.40449,2.22573,0.90391,1.00000
5,duration_Less than an Hour,-1.17488,0.30886,0.07967,1.19733,0.08923,0.87304
6,duration_Between 1 and 2 hours,-0.71561,0.48890,0.21526,1.11038,0.08730,0.87304
7,duration_Between 2 and 3 hours,-0.65067,0.52169,0.28014,0.97152,0.04026,0.48316
8,duration_Between 3 and 4 hours,-0.15420,0.85710,0.46892,1.56661,0.61630,1.00000
9,duration_Between 4 and 5 hours,-0.65284,0.52056,0.26072,1.03938,0.06424,0.70667


## 5 Measurement safeguards

Psychological theory does not turn the survey worry item into GAD-7. The utility below was defined above but has not been applied to any study participant. It requires seven documented items, verified 0–3 coding and the correct two-week timeframe. Missing items yield a missing total. The following artificial fixtures check software only and are not research observations.

Sources: NIH GAD-7 instrument page https://www.nih.gov/node/19876 and SAMHSA 2024 methodological summary, section 3.4.11 https://www.samhsa.gov/data/sites/default/files/reports/rpt56485/2024-nsduh-method-summ-defs/2024-nsduh-method-summ-defs/2024-nsduh-method-summary-defs.htm . Consult the source questionnaire before any use.

In [5]:
columns=[f"item_{i}" for i in range(1,8)]
fixture=pd.DataFrame([[0]*7,[3]*7,[1,1,1,1,1,1,np.nan]],columns=columns)
flags=dict(instrument_verified=True,coding_verified=True,timeframe_verified=True)
scored=score_verified_gad7(fixture,columns,**flags)
assert scored.iloc[0]==0 and scored.iloc[1]==21 and pd.isna(scored.iloc[2])
def must_reject(fn):
    try: fn()
    except (ValueError,TypeError): return
    raise AssertionError("Invalid measurement input was accepted")
must_reject(lambda:score_verified_gad7(fixture,columns))
must_reject(lambda:score_verified_gad7(fixture,columns[:6],**flags))
must_reject(lambda:score_verified_gad7(fixture,columns[:6]+[columns[0]],**flags))
invalid=fixture.copy();invalid.iloc[0,0]=4
must_reject(lambda:score_verified_gad7(invalid,columns,**flags))
invalid.iloc[0,0]=0.5
must_reject(lambda:score_verified_gad7(invalid,columns,**flags))
print("8 scoring assertions passed on artificial fixtures only. Empirical GAD-7 analysis remains NOT EXECUTED.")

8 scoring assertions passed on artificial fixtures only. Empirical GAD-7 analysis remains NOT EXECUTED.


<ipython-input-1-5aedaf105be2>:15: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.5' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  invalid.iloc[0,0]=0.5


## 6 Mediation readiness and honest completion status

The original hypothesis requires perceived personalisation, comparison and anxiety in the same participants. Neither analysed source has them jointly. Therefore no direct, indirect or mediated-percentage estimate is produced. Cross-sectional indirect associations would not establish causality even if the measures existed. Temporal order, pre-exposure confounding, sampling structure and an appropriate outcome model must be addressed before a causal mediation analysis. See Li et al. (2023), https://doi.org/10.1037/tra0001421 .

The Kenyan candidate in Baseke et al. (2026), https://doi.org/10.1016/j.dib.2026.112513 , is **not analysed**: its authors ask prospective analysts to contact them about appropriate ethics approvals and data-sharing agreements. It also does not establish the personalisation exposure required by the approved topic. Do not pool unrelated records or impute a variable that was never measured.

In [6]:
readiness={"approved_title":'The Impact of Perceived Social Media Algorithm Personalisation on Anxiety Levels Among Generation Z: The Mediating Role of Social Comparison',"secondary_only":True,"joint_personalisation_comparison_anxiety_available":False,"target_population_verified":False,"mediation_estimated":False,"empirical_gad7_scored":False,"action_required":"Authorised joint secondary data and institutional scope confirmation; no new primary collection."}
(OUT/"measurement_readiness.json").write_text(json.dumps(readiness,indent=2))
print(json.dumps(readiness,indent=2))

{
  "approved_title": "The Impact of Perceived Social Media Algorithm Personalisation on Anxiety Levels Among Generation Z: The Mediating Role of Social Comparison",
  "secondary_only": true,
  "joint_personalisation_comparison_anxiety_available": false,
  "target_population_verified": false,
  "mediation_estimated": false,
  "empirical_gad7_scored": false,
  "action_required": "Authorised joint secondary data and institutional scope confirmation; no new primary collection."
}


## 7 Download results

Only aggregate audit outputs are included. No raw observations, identifiers or individual predictions are exported.

In [7]:
import zipfile
archive=ROOT/"statistical_audit_results.zip"
allowed=["ordinal_model_fit.csv","all_ordinal_predictors.csv","statistical_audit.json","measurement_readiness.json"]
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for name in allowed:z.write(OUT/name,arcname=name)
print("Aggregate output archive:",archive)
# In Colab, uncomment to download:
# from google.colab import files
# files.download(str(archive))

Aggregate output archive: /workspace/scratch/0822e5d1c7a5/work/final_revision/statistical_audit_results.zip
